In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [3]:
import json
import ollama
from models.metric import CDE, exhaustive_CDE, EF, CDEF

## Load benchmark dataset

In [4]:
def parse_benchmark(path:str):
    
    with open(path) as f:
        dataset = json.load(f)
        dataset = dataset['Data']

    A = []
    B = []
    C = []

    for e in dataset:
        A.append(e['A'])
        B.append(e['B'])
        C.append(e['C'])

    return A,B,C

In [5]:
similarities_path = '../data/external/benchmark/test_similarities.json'
A_sim, B_sim, C_sim = parse_benchmark(similarities_path)

mix_path = '../data/external/benchmark/test_mix.json'
A_mix, B_mix, C_mix = parse_benchmark(mix_path)

types_path = '../data/external/benchmark/test_types.json'
A_type, B_type, C_type = parse_benchmark(types_path)

## Get text embeddings

In [6]:
EMBED_MODEL = 'nomic-embed-text'

def get_text_embedding_using_ollama(text:str):
    
    if text:
        response = ollama.embed(
            model=EMBED_MODEL,
            input=text,
        )

        return response["embeddings"]
    
    return None

In [7]:
def get_embeddings(entity_type:list):
    embed_type = []
    
    for entry in entity_type:
        new_entry = []
        for [entity, type] in entry:
            embed = get_text_embedding_using_ollama(entity)
            new_entry.append([embed, type])
        embed_type.append(new_entry)

    return embed_type

In [8]:
A_sim_embed = get_embeddings(A_sim)
B_sim_embed = get_embeddings(B_sim)
C_sim_embed = get_embeddings(C_sim)

A_type_embed = get_embeddings(A_type)
B_type_embed = get_embeddings(B_type)
C_type_embed = get_embeddings(C_type)

A_mix_embed = get_embeddings(A_mix)
B_mix_embed = get_embeddings(B_mix)
C_mix_embed = get_embeddings(C_mix)


## Test metric

In [9]:
def test_metric(metric_fun, A, B, C, beta = None, optimum_mini=True, verbose = False):

    n_passed = 0
    n_failed = 0
    ids_failed = []

    for i, (a, b, c) in enumerate(zip(A, B, C)):

        if beta is not None:
            s1 = metric_fun(a, a, beta)
            s2 = metric_fun(a, b, beta)
            s3 = metric_fun(a, c, beta)
        else:
            s1 = metric_fun(a, a)
            s2 = metric_fun(a, b)
            s3 = metric_fun(a, c)

        # if verbose: print(f'{a}\t{b}\t{c}')
        if optimum_mini:
            if s1 <= s2 <= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)
        else:
            if s1 >= s2 >= s3:
                if verbose: print(f'TEST PASSED\t{s1}\t{s2}\t{s3}\n')
                n_passed += 1
            else:
                # if verbose: print(f'{a}\t{b}\t{c}')
                if verbose: print(f'TEST FAILED\t{s1}\t{s2}\t{s3}\n')
                n_failed += 1
                ids_failed.append(i)

    return n_passed, n_failed, ids_failed

In [10]:
p, f, id = test_metric(CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tCDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id = test_metric(exhaustive_CDE, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\texh_CDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id = test_metric(EF, A_sim_embed, B_sim_embed, C_sim_embed, verbose=False)
print(f'SIM\tEF\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_sim_embed, B_sim_embed, C_sim_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'SIM\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

SIM	CDE		PASSED: 112	FAILED: 18	IDS FAIL: [2, 6, 12, 14, 15, 16, 17, 24, 25, 35, 37, 53, 57, 65, 68, 82, 96, 107]
SIM	exh_CDE		PASSED: 66	FAILED: 64	IDS FAIL: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 46, 53, 57, 65, 68, 81, 82, 87, 88, 89, 90, 91, 93, 94, 95, 96, 107, 121, 123, 125, 126, 128, 129]
SIM	EF		PASSED: 81	FAILED: 49	IDS FAIL: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 121, 122, 123, 124, 125, 126, 127, 128, 129]
SIM	CDEF-0.0	PASSED: 110	FAILED: 20	IDS FAIL: [2, 6, 12, 14, 15, 16, 17, 24, 25, 35, 37, 53, 57, 65, 68, 82, 90, 92, 96, 107]
SIM	CDEF-0.25	PASSED: 119	FAILED: 11	IDS FAIL: [2, 16, 17, 24, 53, 57, 65, 68, 82, 96, 122]
SIM	CDEF-0.5	PASSED: 120	FAILED: 10	IDS FAIL: [53, 57, 65, 68, 121, 122, 123, 124, 125, 127]
SIM	CDEF-0.75	PASSED: 117	FAILED: 13	IDS

In [11]:
p, f, id= test_metric(CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tCDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id= test_metric(exhaustive_CDE, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\texh_CDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id= test_metric(EF, A_type_embed, B_type_embed, C_type_embed, verbose=False)
print(f'TYPE\tEF\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_type_embed, B_type_embed, C_type_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'TYPE\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

TYPE	CDE		PASSED: 130	FAILED: 0	IDS FAIL: []
TYPE	exh_CDE		PASSED: 130	FAILED: 0	IDS FAIL: []
TYPE	EF		PASSED: 81	FAILED: 49	IDS FAIL: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 121, 122, 123, 124, 125, 126, 127, 128, 129]
TYPE	CDEF-0.0	PASSED: 130	FAILED: 0	IDS FAIL: []
TYPE	CDEF-0.25	PASSED: 124	FAILED: 6	IDS FAIL: [121, 123, 125, 126, 128, 129]
TYPE	CDEF-0.5	PASSED: 124	FAILED: 6	IDS FAIL: [121, 123, 125, 126, 128, 129]
TYPE	CDEF-0.75	PASSED: 124	FAILED: 6	IDS FAIL: [121, 123, 125, 126, 128, 129]
TYPE	CDEF-1.0	PASSED: 123	FAILED: 7	IDS FAIL: [121, 122, 123, 125, 126, 128, 129]
TYPE	CDEF-1.25	PASSED: 121	FAILED: 9	IDS FAIL: [121, 122, 123, 124, 125, 126, 127, 128, 129]
TYPE	CDEF-1.5	PASSED: 121	FAILED: 9	IDS FAIL: [121, 122, 123, 124, 125, 126, 127, 128, 129]
TYPE	CDEF-1.75	PASSED: 121	FAILED: 9	IDS FAIL: [121, 122, 123, 124, 125, 126, 127, 128, 129]
TYPE	CDEF-2.0	PASSED: 121	F

In [12]:
p, f, id= test_metric(CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id= test_metric(exhaustive_CDE, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

p, f, id= test_metric(EF, A_mix_embed, B_mix_embed, C_mix_embed, verbose=False)
print(f'MIX\tCDE\t\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

for beta in range(0, 225, 25):
    beta = beta/100.0
    p, f, id= test_metric(CDEF, A_mix_embed, B_mix_embed, C_mix_embed, beta=beta, optimum_mini=False, verbose=False)
    print(f'MIX\tCDEF-{beta}\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

MIX	CDE		PASSED: 108	FAILED: 22	IDS FAIL: [0, 2, 5, 9, 12, 14, 15, 16, 17, 20, 24, 25, 29, 33, 34, 37, 38, 39, 53, 68, 69, 96]
MIX	CDE		PASSED: 91	FAILED: 39	IDS FAIL: [0, 2, 4, 5, 8, 9, 11, 12, 14, 15, 16, 17, 19, 20, 22, 23, 24, 25, 29, 31, 33, 34, 37, 38, 39, 40, 46, 53, 68, 69, 81, 89, 90, 91, 95, 96, 121, 123, 125]
MIX	CDE		PASSED: 89	FAILED: 41	IDS FAIL: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 14, 15, 16, 17, 19, 20, 22, 23, 24, 25, 29, 31, 32, 33, 34, 35, 37, 38, 39, 84, 86, 112, 119, 120, 121, 122, 123, 125, 127]
MIX	CDEF-0.0	PASSED: 107	FAILED: 23	IDS FAIL: [0, 2, 5, 9, 12, 14, 15, 16, 17, 20, 24, 25, 29, 33, 34, 37, 38, 39, 53, 68, 69, 90, 96]
MIX	CDEF-0.25	PASSED: 109	FAILED: 21	IDS FAIL: [0, 2, 5, 9, 12, 14, 16, 17, 20, 24, 25, 29, 33, 34, 37, 38, 39, 53, 68, 69, 96]
MIX	CDEF-0.5	PASSED: 112	FAILED: 18	IDS FAIL: [0, 5, 9, 12, 14, 16, 17, 20, 25, 29, 33, 34, 37, 38, 39, 53, 68, 69]
MIX	CDEF-0.75	PASSED: 123	FAILED: 7	IDS FAIL: [12, 14, 16, 25, 53, 68, 69]
MIX	CDEF-1.0	PASSED:

## Debugging

In [13]:
from operator import itemgetter 

b = [90,92]
p, f, id= test_metric(
    CDEF, 
    itemgetter(*b)(A_sim_embed), 
    itemgetter(*b)(B_sim_embed), 
    itemgetter(*b)(C_sim_embed), 
    beta=0.0, optimum_mini=False, verbose=False)


print(f'SIM\tCDEF-0\tPASSED: {p}\tFAILED: {f}\tIDS FAIL: {id}')

SIM	CDEF-0	PASSED: 0	FAILED: 2	IDS FAIL: [0, 1]
